# SPFI ×4 — training on synthetic SEM defect data, compared with the EMPS model (Colab GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SS1162/SR_amat_3/blob/riki/synthetic-data/notebooks/04_synthetic_train_colab.ipynb)

Same training as `train_colab.ipynb` (the repo's `algo/` code unchanged), with one difference: the dataset.
Instead of EMPS, the model trains on **synthetic SEM images with defects** made by
`algo/qiu_v2_defect_generator.py` (Qiu et al., CVPR 2026: training-free single-image diffusion) from your real
images. Then both models are compared on the same images:

1. **Your data**: put the real images on Google Drive (layout below).
2. **Hold-out**: `HOLDOUT` of the real images are kept out of the generator. They are the test images of the
   comparison, so neither model has seen them or anything made from them.
3. **Generate** the synthetic images (cached on Drive: the next run skips this step).
4. **Train** with the same width / recipe / budget as the EMPS run (default: C8, `v2`, 10000 iterations), so the
   data is the only difference. Resumes after a disconnect, like `train_colab.ipynb`.
5. **Compare** the EMPS model, the synthetic model and bicubic: PSNR / SSIM on the held-out real images (also
   inside the defect only) and, optionally, on the EMPS validation images.

**Data layout on Drive** (`REAL_DIR`, default `MyDrive/sem_data`):

```
sem_data/
  defects/   real defect images (PNG / TIF / JPG, read as grayscale)
  masks/     one binary mask per defect image, same file name stem (white = defect)
  normals/   defect-free images, the canvases the new defects are drawn into
```

**Before you start:** Runtime → Change runtime type → **GPU**. Then Runtime → Run all.

In [ ]:
import torch

print("torch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Runtime > Change runtime type > GPU, then rerun (the generator is very slow on a CPU).")

## Settings
Keep `EMBED_DIM`, `RECIPE`, `MAX_ITERS` and `SEED` equal to the EMPS run in `EMPS_MODEL`, so the comparison
differs only in the training data. The EMPS model is read from `DRIVE_DIR` (where `train_colab.ipynb` saved it);
if it is not there you are asked to upload it.

In [ ]:
import os
from pathlib import Path

# ---- Code: the generator lives on this branch ----
CODE_REPO_URL = "https://github.com/SS1162/SR_amat_3.git"
CODE_BRANCH = "riki/synthetic-data"

# ---- Your real SEM images on Google Drive: defects/, masks/, normals/ (see the layout above) ----
REAL_DIR = Path("/content/drive/MyDrive/sem_data")
HOLDOUT = 0.2             # share of the defect images and of the normal images kept out of generation (= test images)
HOLDOUT_SEED = 0

# ---- Synthetic data (algo/qiu_v2_defect_generator.py) ----
RATIO = 5                 # synthetic images per real defect image
N_SYNTH = None            # or an exact number of synthetic images (overrides RATIO)
GEN_SEED = 20261018

# ---- Training: the same recipe and budget as the EMPS run in EMPS_MODEL ----
RECIPE = "v2"
if RECIPE == "v1":
    BATCH_SIZE, LR, LR_SCHEDULE, WARMUP_ITERS, EMA_DECAY, AUGMENT = 1, 1e-4, "constant", 0, None, False
elif RECIPE == "v2":
    BATCH_SIZE, LR, LR_SCHEDULE, WARMUP_ITERS, EMA_DECAY, AUGMENT = 16, 2e-4, "cosine", 500, 0.999, True
else:
    raise ValueError(f"unknown RECIPE {RECIPE!r}")
WEIGHT_DECAY = 1e-2
PATIENCE = 1000           # 1000 = early stopping off (fixed budget)
SEED = 0
SPLIT_SEED = 0            # the trainer's 80/20 train/val split of the synthetic images
EMBED_DIM = 8
MAX_ITERS = 10000

# ---- Comparison ----
DRIVE_DIR = Path("/content/drive/MyDrive/spfi_checkpoints")
EMPS_MODEL = "spfi_x4_final_e8_s0_v2_it10000.pt"   # the EMPS-trained model to compare with, in DRIVE_DIR
COMPARE_ON_EMPS = True    # also score both models on the EMPS validation images (downloads EMPS, ~1 min)
N_GALLERY = 6             # held-out real images shown side by side
TILE, OVERLAP, BATCH_TILES = 64, 16, 16            # same tiling as the app
SHAVE = 4                 # HR border pixels ignored by PSNR / SSIM
CROP = 128                # side of the zoomed region (HR pixels)
RESUME_EVERY = 500        # training state saved to Drive every N iterations (resume after a disconnect)

# ---- Public EMPS release (same as train_colab.ipynb), only for COMPARE_ON_EMPS ----
EMPS_COMMIT = "dd8f86d714894fd08f3db80041893982ce058dd6"
EMPS_URL = f"https://codeload.github.com/by256/emps/zip/{EMPS_COMMIT}"
EMPS_SHA256 = "141462d290f9ac780bf9599ef5cb4b3e6708adfe6abafd1d4b9d6c786a6ad3f8"

IN_COLAB = "google.colab" in __import__("sys").modules
WORK_DIR = Path("/content") if IN_COLAB else Path(os.environ.get("SR_WORK_DIR", Path.cwd() / "_synthetic"))
CODE_DIR, OUTPUT_DIR = WORK_DIR / "SR_amat_3", WORK_DIR / "output"
# Local testing only: environment variables override the values above.
CODE_REPO_URL = os.environ.get("SR_REPO_URL") or CODE_REPO_URL
CODE_BRANCH = os.environ.get("SR_REF") or CODE_BRANCH
REAL_DIR = Path(os.environ.get("SR_REAL_DIR") or REAL_DIR)
DRIVE_DIR = Path(os.environ.get("SR_DRIVE_DIR") or DRIVE_DIR)
MAX_ITERS = int(os.environ.get("SR_MAX_ITERS") or MAX_ITERS)
BATCH_SIZE = int(os.environ.get("SR_BATCH") or BATCH_SIZE)
N_SYNTH = int(os.environ["SR_N_SYNTH"]) if os.environ.get("SR_N_SYNTH") else N_SYNTH
MAX_VAL = int(os.environ["SR_MAX_VAL"]) if os.environ.get("SR_MAX_VAL") else None
RESUME_EVERY = int(os.environ.get("SR_RESUME_EVERY") or RESUME_EVERY)
assert LR_SCHEDULE != "cosine" or MAX_ITERS is not None, "the cosine schedule needs MAX_ITERS (it decays to it)"
RUN_TAG = f"syn_e{EMBED_DIM}_s{SEED}_{RECIPE}_it{MAX_ITERS}"   # -> spfi_x4_final_syn_e8_s0_v2_it10000.pt
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Code and dependencies

In [ ]:
import subprocess
import sys

if IN_COLAB:
    # algo/ needs pytorch_wavelets; the generator needs Qiu et al.'s efficient_sid (public, MIT License)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytorch_wavelets>=1.3", "PyWavelets>=1.6",
                    "setuptools<81", "git+https://github.com/haojunqiu/efficient-SID-code"], check=True)

In [ ]:
import time


def git(*args, cwd=None, attempts=3):
    """Run git, retrying network hiccups."""
    for attempt in range(1, attempts + 1):
        result = subprocess.run(["git", *args], cwd=cwd, text=True, capture_output=True)
        if result.returncode == 0:
            return result.stdout.strip()
        if attempt < attempts:
            time.sleep(5 * attempt)
    raise RuntimeError(f"git {args[0]} failed:\n{result.stderr}")


if not (CODE_DIR / ".git").exists():
    git("clone", "--quiet", "--no-checkout", CODE_REPO_URL, str(CODE_DIR))
git("fetch", "--quiet", "origin", CODE_BRANCH, cwd=CODE_DIR)
git("checkout", "--quiet", "--force", "--detach", "FETCH_HEAD", cwd=CODE_DIR)
print("code:", CODE_BRANCH, "@", git("log", "-1", "--format=%h %s", cwd=CODE_DIR))
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
for name in [m for m in sys.modules if m == "algo" or m.startswith("algo.")]:
    del sys.modules[name]   # re-import algo/ from the branch just checked out

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

## Your data and the hold-out split
`HOLDOUT` of the defect images and of the normal images are set aside **before** generation. The generator only
sees the rest, so the held-out images are fair test images for both models. A folder with a single image is not
split (that image is used for generation and there is no test image from that folder).

In [ ]:
import hashlib
import random
import shutil

import numpy as np
from PIL import Image

from algo import qiu_v2_defect_generator as gen


def images_in(folder):
    return sorted(p for p in folder.iterdir() if p.suffix.lower() in gen.IMAGE_EXT)


def holdout_split(paths):
    """(used for generation, held out for testing), chosen at random with HOLDOUT_SEED."""
    if len(paths) < 2:
        return paths, []
    shuffled = sorted(paths)
    random.Random(HOLDOUT_SEED).shuffle(shuffled)
    n_test = min(len(paths) - 1, max(1, round(HOLDOUT * len(paths))))
    return sorted(shuffled[n_test:]), sorted(shuffled[:n_test])


for sub in ("defects", "masks", "normals"):
    assert (REAL_DIR / sub).is_dir(), f"missing folder {REAL_DIR / sub} (see the data layout at the top)"
gen_defects, test_defects = holdout_split(images_in(REAL_DIR / "defects"))
gen_normals, test_normals = holdout_split(images_in(REAL_DIR / "normals"))
assert gen_defects and gen_normals, "need at least one defect image (with mask) and one defect-free image"

# The generator reads folders: copy the images it may use (and their masks) into GEN_INPUT.
GEN_INPUT = WORK_DIR / "gen_input"
shutil.rmtree(GEN_INPUT, ignore_errors=True)
for sub in ("defects", "masks", "normals"):
    (GEN_INPUT / sub).mkdir(parents=True)
for p in gen_defects:
    shutil.copy2(p, GEN_INPUT / "defects" / p.name)
    m = gen.find_mask(REAL_DIR / "masks", p.stem)
    shutil.copy2(m, GEN_INPUT / "masks" / m.name)
for p in gen_normals:
    shutil.copy2(p, GEN_INPUT / "normals" / p.name)

# Test images: (name, path, defect mask or None)
TEST = ([(f"defect_{p.stem}", p, gen.gray(gen.find_mask(REAL_DIR / "masks", p.stem)) > gen.MASK_THRESHOLD)
         for p in test_defects] + [(f"normal_{p.stem}", p, None) for p in test_normals])

sizes = [gen.gray(p).shape for p in gen_defects + gen_normals]
print(f"generation: {len(gen_defects)} defect images, {len(gen_normals)} defect-free images")
print(f"held out for the comparison: {len(test_defects)} defect images, {len(test_normals)} defect-free images")
print(f"image sizes (H x W): {sorted(set(sizes))[:5]}{' ...' if len(set(sizes)) > 5 else ''}")
if min(min(s) for s in sizes) < 256:
    print("NOTE: images smaller than 256 px are upscaled to 256 for training (the HR patch size).")

# Identifies this exact input + settings, so the cached synthetic set is reused only when nothing changed.
fingerprint = hashlib.sha1(repr(
    ([(p.name, p.stat().st_size) for p in gen_defects + gen_normals], RATIO, N_SYNTH, GEN_SEED)).encode()).hexdigest()[:8]
SYN_TAG = f"{fingerprint}_" + (f"n{N_SYNTH}" if N_SYNTH is not None else f"r{RATIO}")
print("synthetic set:", SYN_TAG)

## Generate the synthetic images
Each new defect takes the shape of a real defect mask (scaled, elastically deformed) and its pixels are generated
from the patches of the real defect images, then drawn into a defect-free image. The finished set is zipped to
`DRIVE_DIR/synthetic_<tag>.zip`; a later run with the same input and settings unzips it instead of generating
again. The `kept` count can be lower than `made`: results fainter than the generator's contrast threshold are
dropped.

In [ ]:
import json
import zipfile

SYN_DIR = WORK_DIR / "synthetic" / SYN_TAG
SYN_ZIP = DRIVE_DIR / f"synthetic_{SYN_TAG}.zip"

if not (SYN_DIR / "done.json").exists() and SYN_ZIP.exists():
    print(f"unzipping {SYN_ZIP.name} (made in an earlier run)")
    with zipfile.ZipFile(SYN_ZIP) as zf:
        zf.extractall(SYN_DIR)
if not (SYN_DIR / "done.json").exists():
    t0 = time.time()
    gen.SEED = GEN_SEED
    gen.make(GEN_INPUT / "defects", GEN_INPUT / "masks", GEN_INPUT / "normals", SYN_DIR, RATIO, N_SYNTH)
    print(f"generated in {(time.time() - t0) / 60:.1f} min")
    tmp = shutil.make_archive(str(WORK_DIR / SYN_ZIP.stem), "zip", SYN_DIR)
    shutil.copy2(tmp, SYN_ZIP)   # write locally, then copy: a disconnect mid-copy leaves no half file
    print("saved to", SYN_ZIP)

done = json.loads((SYN_DIR / "done.json").read_text())
syn_images = sorted((SYN_DIR / "images").glob("*.png"))
print(f"{len(syn_images)} synthetic images (made {done['made']}, kept {done['kept']}) in {SYN_DIR / 'images'}")
assert syn_images, "the generator kept no image: see samples.csv in the synthetic folder"

In [ ]:
import matplotlib.pyplot as plt

# A few synthetic images, with the outline of the new defect
show_n = min(6, len(syn_images))
fig, axes = plt.subplots(1, show_n, figsize=(3 * show_n, 3.2), squeeze=False)
for ax, p in zip(axes[0], syn_images[:show_n]):
    ax.imshow(np.asarray(Image.open(p)), cmap="gray", vmin=0, vmax=255)
    ax.contour(np.asarray(Image.open(SYN_DIR / "masks" / p.name)) > 127, levels=[0.5], colors="#FFD400",
               linewidths=0.8)
    ax.set_title(p.stem, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
plt.suptitle("synthetic images (yellow = new defect)")
plt.tight_layout()
plt.show()

## Train on the synthetic images
Exactly as in `train_colab.ipynb`, with `emps_dir` pointing to the synthetic set (the trainer reads
`<dir>/images/*.png` and splits it 80/20 into training / validation). The training state is saved to Drive every
`RESUME_EVERY` iterations: after a disconnect, Run all again with the same settings and training continues.

In [ ]:
from datetime import datetime

from algo.train import SPFITrainer

stamp = f"{RUN_TAG}_{datetime.now().strftime('%Y%m%d-%H%M%S')}"
CKPT_DIR = WORK_DIR / "checkpoints" / stamp
RUN_DIR = WORK_DIR / "runs" / stamp

trainer = SPFITrainer(
    emps_dir=SYN_DIR,
    ckpt_dir=CKPT_DIR,
    lr=LR,
    wd=WEIGHT_DECAY,
    patience=PATIENCE,
    seed=SEED,
    split_seed=SPLIT_SEED,
    embed_dim=EMBED_DIM,
    run_dir=RUN_DIR,
    batch_size=BATCH_SIZE,
    lr_schedule=LR_SCHEDULE,
    warmup_iters=WARMUP_ITERS,
    total_iters=MAX_ITERS,
    ema_decay=EMA_DECAY,
    augment=AUGMENT,
    cache_images=True,
)
print(f"embed_dim {EMBED_DIM}: {sum(p.numel() for p in trainer.model.parameters()):,} parameters")

RUN_SETTINGS = {"data": SYN_TAG, "embed_dim": EMBED_DIM, "seed": SEED, "split_seed": SPLIT_SEED, "lr": LR,
                "wd": WEIGHT_DECAY, "patience": PATIENCE, "max_iters": MAX_ITERS, "recipe": RECIPE,
                "batch_size": BATCH_SIZE, "lr_schedule": LR_SCHEDULE, "warmup_iters": WARMUP_ITERS,
                "ema_decay": EMA_DECAY, "augment": AUGMENT}
RESUME_FILE = DRIVE_DIR / f"resume_{RUN_TAG}_{SYN_TAG}.pt"
RESUMED_FROM, PRIOR_SECONDS = 0, 0.0


def save_resume(elapsed_seconds):
    """Everything needed to continue exactly where training is now (called between iterations)."""
    best = CKPT_DIR / "spfi_x4_best.pt"
    state = {
        "settings": RUN_SETTINGS,
        "iteration": trainer.iteration,
        "model_state": trainer.model.state_dict(),
        "optimizer_state": trainer.optimizer.state_dict(),
        "ema_state": trainer.ema_model.state_dict() if trainer.ema_model is not None else None,
        "best_val_loss": trainer.best_val_loss,
        "no_improve": trainer.no_improve,
        "best_checkpoint": torch.load(best, map_location="cpu", weights_only=True) if best.exists() else None,
        "train_rng": trainer._train_rng.getstate(),
        "torch_rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        "elapsed_seconds": elapsed_seconds,
    }
    tmp = WORK_DIR / f"{RESUME_FILE.name}.tmp"
    torch.save(state, tmp)
    shutil.copy2(tmp, RESUME_FILE)


if RESUME_FILE.exists():
    state = torch.load(RESUME_FILE, map_location=trainer.device, weights_only=False)
    assert state["settings"] == RUN_SETTINGS, f"{RESUME_FILE.name} was saved with other settings: {state['settings']}"
    trainer.model.load_state_dict(state["model_state"])
    trainer.optimizer.load_state_dict(state["optimizer_state"])
    if state.get("ema_state") is not None:
        trainer.ema_model.load_state_dict(state["ema_state"])
    trainer.iteration = state["iteration"]
    trainer.best_val_loss = state["best_val_loss"]
    trainer.no_improve = state["no_improve"]
    if state["best_checkpoint"] is not None:
        torch.save(state["best_checkpoint"], CKPT_DIR / "spfi_x4_best.pt")
    trainer._train_rng.setstate(state["train_rng"])
    torch.set_rng_state(state["torch_rng"].cpu())
    if state["cuda_rng"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([s.cpu() for s in state["cuda_rng"]])
    RESUMED_FROM, PRIOR_SECONDS = state["iteration"], state["elapsed_seconds"]
    print(f"RESUMED from {RESUME_FILE.name}: iteration {RESUMED_FROM}, {PRIOR_SECONDS / 60:.1f} min already trained")
else:
    print(f"no resume file ({RESUME_FILE.name}): starting from iteration 0")

In [ ]:
import threading

stop_event = threading.Event()
train_started = time.time()
try:
    # Chunks of RESUME_EVERY iterations (the same updates as one long call); the state goes to Drive in between.
    while not stop_event.is_set() and trainer.iteration < MAX_ITERS:
        target = min(trainer.iteration + RESUME_EVERY, MAX_ITERS)
        trainer.train(stop_event=stop_event, max_iters=target)
        if trainer.iteration < target:   # early stopping or the stop button
            break
        save_resume(PRIOR_SECONDS + time.time() - train_started)
        print(f"[resume] state saved at iteration {trainer.iteration}")
finally:
    train_seconds = PRIOR_SECONDS + time.time() - train_started
    PRIOR_SECONDS = train_seconds   # running this cell again (after Stop) keeps counting from here
print(f"finished at iteration {trainer.iteration}, best val loss {trainer.best_val_loss:.4f}, {train_seconds / 60:.1f} min")
if trainer.iteration < MAX_ITERS:
    print(f"\n*** WARNING: stopped at iteration {trainer.iteration} of {MAX_ITERS}: not the same budget as the EMPS "
          "model. Run the notebook again with the same settings to resume. ***\n")

## Export the weights
`spfi_x4_final_syn_<tag>.pt` (the weights after exactly `MAX_ITERS`, used in the comparison) and
`spfi_x4_best_syn_<tag>.pt` go to `DRIVE_DIR`, in the format the local app loads.

In [ ]:
from algo.model import build_spfi

trainer._save_checkpoint("final")
EXPORTS = {}
for kind in ("best", "final"):
    src = CKPT_DIR / f"spfi_x4_{kind}.pt"
    if not src.exists():
        continue
    dst = OUTPUT_DIR / f"spfi_x4_{kind}_{RUN_TAG}.pt"
    shutil.copy2(src, dst)
    ckpt = torch.load(dst, map_location="cpu", weights_only=True)
    build_spfi(scale=ckpt["hparams"]["scale"], embed_dim=ckpt["hparams"]["embed_dim"]).load_state_dict(ckpt["model_state"])
    shutil.copy2(dst, DRIVE_DIR / dst.name)
    EXPORTS[kind] = dst
    print(f"{dst.name}: iteration {ckpt['iteration']} -> {DRIVE_DIR}")

train_info = {"run_tag": RUN_TAG, "synthetic_set": SYN_TAG, "synthetic_images": len(syn_images),
              "generation": done, "code_branch": CODE_BRANCH, "code_commit": git("rev-parse", "HEAD", cwd=CODE_DIR),
              **RUN_SETTINGS, "iterations": trainer.iteration, "train_seconds": round(train_seconds, 1),
              "held_out": [name for name, _, _ in TEST],
              "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
(OUTPUT_DIR / f"train_info_{RUN_TAG}.json").write_text(json.dumps(train_info, indent=2))
shutil.copy2(OUTPUT_DIR / f"train_info_{RUN_TAG}.json", DRIVE_DIR)

## Compare: EMPS model vs synthetic model
Both models (and bicubic) get the same inputs: each held-out real image is degraded the way the models were
trained (`degrade`: Gaussian blur + bicubic ×4 downscale), reconstructed with the app's 64×64 tiling, and compared
with the original. For the held-out **defect** images, PSNR is also measured **inside the defect** (the mask, grown
by 4 px), since that is what the synthetic data is about.

In [ ]:
import pandas as pd
import torch.nn.functional as F
from scipy import ndimage
from torchvision.transforms.functional import to_tensor

from algo.image_loading import load_image
from algo.preprocessing import normalize_image
from algo.train import degrade, psnr, ssim_loss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SCALE = 4


def load_model(path):
    ckpt = torch.load(path, map_location=device, weights_only=True)
    model = build_spfi(scale=ckpt["hparams"]["scale"], embed_dim=ckpt["hparams"]["embed_dim"])
    model.load_state_dict(ckpt["model_state"])
    return model.to(device).eval(), ckpt


emps_path = DRIVE_DIR / EMPS_MODEL
if not emps_path.exists() and IN_COLAB:
    from google.colab import files
    print(f"{EMPS_MODEL} is not in {DRIVE_DIR}: upload the EMPS-trained model (.pt)")
    name, data = next(iter(files.upload().items()))
    emps_path = WORK_DIR / name
    emps_path.write_bytes(data)
MODELS = {}
for label, path in (("EMPS model", emps_path), ("synthetic model", EXPORTS["final"])):
    if not Path(path).exists():
        print(f"WARNING: {path} not found: '{label}' is left out of the comparison")
        continue
    MODELS[label], ckpt = load_model(path)
    print(f"{label}: {Path(path).name}, C{ckpt['hparams']['embed_dim']}, iteration {ckpt.get('iteration')}")


def tile_starts(n):
    if n <= TILE:
        return [0]
    return list(range(0, n - TILE, TILE - OVERLAP)) + [n - TILE]


@torch.no_grad()
def super_resolve(model, lr):
    # [1, 1, h, w] LR -> [1, 1, h*SCALE, w*SCALE], overlapping 64x64 tiles averaged (as in the app)
    _, _, h, w = lr.shape
    lr = F.pad(lr.to(device), (0, max(0, TILE - w), 0, max(0, TILE - h)), mode="replicate")
    H, W = lr.shape[-2:]
    coords = [(y, x) for y in tile_starts(H) for x in tile_starts(W)]
    out = torch.zeros(1, 1, H * SCALE, W * SCALE, device=device)
    hits = torch.zeros_like(out)
    t = TILE * SCALE
    for i in range(0, len(coords), BATCH_TILES):
        chunk = coords[i:i + BATCH_TILES]
        sr = model(torch.cat([lr[..., y:y + TILE, x:x + TILE] for y, x in chunk]))
        for (y, x), tile in zip(chunk, sr):
            out[..., y * SCALE:y * SCALE + t, x * SCALE:x * SCALE + t] += tile
            hits[..., y * SCALE:y * SCALE + t, x * SCALE:x * SCALE + t] += 1
    return (out / hits)[..., :h * SCALE, :w * SCALE].clamp(0, 1).cpu()


def bicubic(lr):
    return F.interpolate(lr, scale_factor=SCALE, mode="bicubic", align_corners=False).clamp(0, 1)


def load_pair(path, name):
    # HR [1,1,H,W] cropped to a multiple of SCALE, and its LR made with the training degradation
    hr = to_tensor(normalize_image(load_image(path)))
    _, H, W = hr.shape
    hr = hr[:, :H - H % SCALE, :W - W % SCALE]
    return hr.unsqueeze(0), degrade(hr, filename=name, scale=SCALE).unsqueeze(0)


shave = lambda t: t[..., SHAVE:-SHAVE, SHAVE:-SHAVE] if SHAVE else t
metrics = lambda out, hr: (psnr(shave(out), shave(hr)), 1 - ssim_loss(shave(out), shave(hr)).item())


def masked_psnr(out, hr, mask):
    mse = float(((out - hr)[0, 0][mask] ** 2).mean())
    return float("inf") if mse == 0 else 10 * np.log10(1 / mse)


def score(items):
    """items: (name, path, mask or None) -> one row per image, with every method's PSNR / SSIM."""
    rows = []
    for n, (name, path, mask) in enumerate(items, 1):
        hr, lr = load_pair(path, name)
        rec = {"image": name}
        if mask is not None:
            grown = torch.from_numpy(ndimage.binary_dilation(mask, iterations=4)[:hr.shape[-2], :hr.shape[-1]])
        for label, out in [("bicubic", bicubic(lr)), *((l, super_resolve(m, lr)) for l, m in MODELS.items())]:
            rec[f"{label}_psnr"], rec[f"{label}_ssim"] = metrics(out, hr)
            if mask is not None:
                rec[f"{label}_defect_psnr"] = masked_psnr(out, hr, grown)
        rows.append(rec)
        if n % 10 == 0 or n == len(items):
            print(f"{n}/{len(items)} images")
    return pd.DataFrame(rows).set_index("image")


def summarize(scores, title):
    out = []
    for label in ["bicubic", *MODELS]:
        p = scores[f"{label}_psnr"]
        row = {"method": label, "PSNR (dB)": p.mean(), "SSIM": scores[f"{label}_ssim"].mean(),
               "Δ PSNR vs bicubic": p.mean() - scores["bicubic_psnr"].mean()}
        if f"{label}_defect_psnr" in scores:
            row["PSNR inside defect"] = scores[f"{label}_defect_psnr"].mean()
        if label != "bicubic":
            row["beats bicubic"] = f"{int((p > scores['bicubic_psnr']).sum())}/{len(p)}"
        out.append(row)
    table = pd.DataFrame(out).set_index("method")
    print(f"\n{title} ({len(scores)} images)")
    display(table.round(3))
    if len(MODELS) == 2:
        a, b = (scores[f"{m}_psnr"] for m in MODELS)
        print(f"synthetic model better than the EMPS model on {int((b > a).sum())}/{len(b)} images "
              f"(mean ΔPSNR {(b - a).mean():+.2f} dB)")
    return table


assert TEST, "no held-out real image (each folder had a single image): set HOLDOUT or add images"
REAL_SCORES = score(TEST)
REAL_SCORES.to_csv(OUTPUT_DIR / "real_heldout_scores.csv")
REAL_SUMMARY = summarize(REAL_SCORES, "held-out real SEM images (never seen by the generator or either model)")

In [ ]:
# The EMPS validation images (20% of EMPS train.csv, never trained on by the EMPS model): what the synthetic model
# gives up on general microscopy images.
if COMPARE_ON_EMPS:
    import urllib.request

    from algo.train import split_files

    archive = WORK_DIR / "emps.zip"
    if not archive.exists():
        print("Downloading EMPS...")
        urllib.request.urlretrieve(EMPS_URL, archive)
    if hashlib.sha256(archive.read_bytes()).hexdigest() != EMPS_SHA256:
        print("WARNING: EMPS archive SHA-256 differs from the recorded one")
    emps_dir = WORK_DIR / "emps_train" / "images"
    emps_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as zf:
        root = next(n for n in zf.namelist() if n.endswith("/train.csv"))[:-len("train.csv")]
        train_ids = [l.split(",")[0].strip() for l in zf.read(root + "train.csv").decode().splitlines() if l.strip()]
        for i in train_ids:
            if not (emps_dir / f"{i}.png").exists():
                (emps_dir / f"{i}.png").write_bytes(zf.read(f"{root}images/{i}.png"))
    _, val_ids = split_files(train_ids, 0)   # SPLIT_SEED 0 of the EMPS runs
    val_ids = val_ids[:MAX_VAL] if MAX_VAL else val_ids
    EMPS_SCORES = score([(i, emps_dir / f"{i}.png", None) for i in val_ids])
    EMPS_SCORES.to_csv(OUTPUT_DIR / "emps_val_scores.csv")
    EMPS_SUMMARY = summarize(EMPS_SCORES, "EMPS validation images")
else:
    print("COMPARE_ON_EMPS = False: skipped")

## Gallery: held-out real images
LR input, bicubic, the EMPS model, the synthetic model and the original. The zoom is centred on the defect (on
defect images) or on the centre of the image.

In [ ]:
import re

from matplotlib.patches import Rectangle

GALLERY_DIR = OUTPUT_DIR / "gallery"
GALLERY_DIR.mkdir(parents=True, exist_ok=True)
order = REAL_SCORES["bicubic_psnr"].sort_values().index.tolist()   # hardest -> easiest for bicubic
picked = list(dict.fromkeys(order[round(q * (len(order) - 1))] for q in np.linspace(0, 1, min(N_GALLERY, len(order)))))
by_name = {name: (path, mask) for name, path, mask in TEST}

for n, name in enumerate(picked, 1):
    path, mask = by_name[name]
    hr, lr = load_pair(path, name)
    H, W = hr.shape[-2:]
    k = min(CROP, H, W)
    cy, cx = ndimage.center_of_mass(mask[:H, :W]) if mask is not None and mask[:H, :W].any() else (H / 2, W / 2)
    y, x = int(min(max(cy - k / 2, 0), H - k)), int(min(max(cx - k / 2, 0), W - k))
    outs = {"bicubic": bicubic(lr), **{label: super_resolve(m, lr) for label, m in MODELS.items()}}
    panels = ([("LR input (×4, no smoothing)", F.interpolate(lr, scale_factor=SCALE, mode="nearest"), "")]
              + [(lbl, img, "PSNR {:.2f}  SSIM {:.3f}".format(*metrics(img, hr))) for lbl, img in outs.items()]
              + [("original (HR)", hr, "")])
    fig, axes = plt.subplots(2, len(panels), figsize=(2.9 * len(panels), 6.6))
    for col, (title, img, cap) in enumerate(panels):
        axes[0, col].imshow(img[0, 0], cmap="gray", vmin=0, vmax=1)
        axes[0, col].add_patch(Rectangle((x, y), k, k, fill=False, ec="#FFD400", lw=1.3))
        axes[0, col].set_title(title + ("\n" + cap if cap else ""), fontsize=9)
        axes[1, col].imshow(img[0, 0, y:y + k, x:x + k], cmap="gray", vmin=0, vmax=1, interpolation="nearest")
        if mask is not None and col == len(panels) - 1:
            axes[1, col].contour(mask[y:y + k, x:x + k], levels=[0.5], colors="#FFD400", linewidths=0.8)
        axes[1, col].set_title("zoom", fontsize=8)
    for ax in axes.flat:
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle(f"[{n}/{len(picked)}] {name} — held-out real image", fontsize=11)
    plt.tight_layout()
    plt.savefig(GALLERY_DIR / f"{n:02d}_{re.sub(r'[^A-Za-z0-9]+', '_', name)}.png", dpi=100)
    plt.show()

## Download the results

In [ ]:
results_zip = shutil.make_archive(str(WORK_DIR / "synthetic_vs_emps_results"), "zip", OUTPUT_DIR)
shutil.copy2(results_zip, DRIVE_DIR)
print("saved:", results_zip, "and in", DRIVE_DIR)
if IN_COLAB:
    from google.colab import files
    files.download(results_zip)

## Reading the results
* **The main question** is the first table: on real SEM images that neither the generator nor either model has seen,
  does the synthetic model get a higher PSNR / SSIM than the EMPS model, and on how many images?
  *PSNR inside defect* shows the same for the defect region alone.
* **The EMPS table** shows what the synthetic model gives up on general microscopy images. A model trained on
  one kind of image usually gets better on that kind and worse on others.
* Few held-out images means noisy averages: look at the per-image count ("better on x/y images") and the zoomed
  crops, not only the mean. PSNR favours smooth outputs.
* Both models learned the same degradation (Gaussian blur + bicubic ×4); a real low-resolution SEM image can
  look different.